# Analyse de l’accessibilité des gares en Île-de-France

Ce notebook analyse l’accessibilité des gares franciliennes à partir des fichiers CSV et GEOJSON fournis.

## 1. Charger et explorer les données CSV

Nous allons importer pandas, charger le fichier CSV, et afficher les premières lignes pour comprendre la structure des données.

In [ ]:
import pandas as pd

# Charger le fichier CSV (séparateur ;)
df = pd.read_csv('../data/raw/accessibilite-en-gare.csv', sep=';')

# Afficher les premières lignes et les infos générales
df.head()

In [ ]:
# Afficher les informations générales du DataFrame
df.info()

## 2. Nettoyer et préparer les données

Nous allons séparer la colonne des coordonnées en latitude et longitude, convertir en float, et vérifier les valeurs manquantes.

In [ ]:
# Séparer la colonne 'stop_point_geopoint' en 'lat' et 'lon'
df[['lat', 'lon']] = df['stop_point_geopoint'].str.split(',', expand=True)

df['lat'] = df['lat'].astype(float)
df['lon'] = df['lon'].astype(float)

df[['lat', 'lon']].head()

In [ ]:
# Vérifier les valeurs manquantes
df.isnull().sum()

In [ ]:
# Sauvegarder le DataFrame nettoyé pour usage ultérieur
df.to_csv('../data/processed/cleaned_data.csv', index=False)

## 3. Analyser la répartition de l’accessibilité

Comptons le nombre de gares par niveau d’accessibilité.

In [ ]:
# Répartition des niveaux d’accessibilité
df['accessibility_level_name'].value_counts()

## 4. Calculer les pourcentages d’accessibilité

Calculons le pourcentage de chaque niveau d’accessibilité.

In [ ]:
# Pourcentage de chaque niveau d’accessibilité
df['accessibility_level_name'].value_counts(normalize=True) * 100

## 5. Lister les gares non accessibles

Filtrons les gares dont le niveau d’accessibilité est « Non accessible ».

In [ ]:
# Gares non accessibles
df_non_access = df[df['accessibility_level_name'] == 'Non accessible']
df_non_access[['stop_name', 'accessibility_level_name', 'lat', 'lon']]

## 6. Identifier les zones problématiques

Analysons la distribution géographique des gares non accessibles, par exemple par ligne ou secteur.

In [ ]:
# Exemple : nombre de gares non accessibles par ligne
if 'ligne' in df.columns:
    df_non_access['ligne'].value_counts()
else:
    print("Colonne 'ligne' non disponible dans ce dataset.")

## 7. Exporter les données nettoyées pour Power BI

Les données nettoyées sont enregistrées dans le dossier `processed/` pour une utilisation dans Power BI.

In [ ]:
# (Déjà fait plus haut) Vérification du fichier exporté
import os
os.path.exists('../data/processed/cleaned_data.csv')

In [ ]:
import geopandas as gpd

gdf = gpd.read_file('../data/raw/accessibilite-en-gare.geojson')
gdf.head()

In [ ]:
# Afficher un aperçu des données géospatiales
gdf.plot(figsize=(10, 8), color='lightblue', edgecolor='black')

## 9. Nombre d’arrêts par ligne

Comptons le nombre total de gares par ligne pour identifier les lignes les plus desservies.

In [ ]:
# Nombre total d’arrêts par ligne
if 'ligne' in df.columns:
    arrets_par_ligne = df['ligne'].value_counts()
    display(arrets_par_ligne)
else:
    print("Colonne 'ligne' non disponible dans ce dataset.")

## 10. Top lignes les plus desservies

Affichons les 5 lignes ayant le plus grand nombre de gares.

In [ ]:
# Top 5 des lignes les plus desservies
if 'ligne' in df.columns:
    top_lignes = arrets_par_ligne.head(5)
    display(top_lignes)
else:
    print("Colonne 'ligne' non disponible dans ce dataset.")

## 11. Zones mal couvertes

Identifions les zones géographiques avec peu ou pas de gares accessibles (analyse simple par quadrillage).

In [ ]:
# Quadrillage simple pour repérer les zones mal couvertes
import numpy as np
import matplotlib.pyplot as plt

if 'lat' in df.columns and 'lon' in df.columns:
    # Définir les bornes de la carte
    lat_min, lat_max = df['lat'].min(), df['lat'].max()
    lon_min, lon_max = df['lon'].min(), df['lon'].max()
    
    # Créer une grille
    n_bins = 20
    grid, xedges, yedges = np.histogram2d(df['lat'], df['lon'], bins=n_bins)
    
    # Afficher la densité de gares
    plt.figure(figsize=(8, 6))
    plt.imshow(grid.T, origin='lower', extent=[lat_min, lat_max, lon_min, lon_max], aspect='auto', cmap='Reds')
    plt.colorbar(label='Nombre de gares')
    plt.title('Densité de gares par zone (quadrillage)')
    plt.xlabel('Latitude')
    plt.ylabel('Longitude')
    plt.show()
else:
    print("Colonnes 'lat' et 'lon' non disponibles dans ce dataset.")